In [ ]:
import cv2
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader


#DATA_PATH should have a folder of patches and folder of masks
#patches should be named as desired i.e patch_1, matching mask names should be desiredname_mask i.e. patch_1_mask
DATA_PATH = "FOLDER DIRECTORY"



class TiDataset(Dataset):
    def __init__(self, root_dir): 
        self.images_dir = os.path.join(root_dir, "patches")
        self.masks_dir = os.path.join(root_dir, "masks")
        self.ids = [f.replace("_mask.png", "") for f in os.listdir(self.masks_dir) if f.endswith("_mask.png")]
        
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, i):
        img_name = self.ids[i] + ".jpg"
        mask_name = self.ids[i] + "_mask.png"

        # 1. Load Image
        image_path = os.path.join(self.images_dir, img_name)
        image = cv2.imread(image_path)
        
        if image is None:
             img_name = self.ids[i] + ".png"
             image_path = os.path.join(self.images_dir, img_name)
             image = cv2.imread(image_path)
             
        if image is None:
            raise RuntimeError(f"Could not read image file for ID: {self.ids[i]} at {image_path}")
        
        # Convert to RGB
        if image.ndim == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. Load Mask
        mask_path = os.path.join(self.masks_dir, mask_name)
        mask = cv2.imread(mask_path, 0) # 0 = cv2.IMREAD_GRAYSCALE
        
        if mask is None:
            raise RuntimeError(f"Could not read mask file: {mask_path}")
        
        # --- RESIZE ---
        image = cv2.resize(image, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

        # 3. Format for PyTorch
        # Image: (H, W, C) -> (C, H, W), Normalized 0-1
        image = image.transpose(2, 0, 1).astype('float32') / 255.0
        
        # Mask: Binary 0 or 1, Shape (1, H, W)
        mask = (mask > 128).astype('float32')
        mask = np.expand_dims(mask, 0)
        
        #Convert numpy arrays to PyTorch tensors explicitly
        image_tensor = torch.from_numpy(image)
        mask_tensor = torch.from_numpy(mask)
        
        return image_tensor, mask_tensor

# Re-initialize the dataset
if __name__ == "__main__":

    
    if not os.path.exists(DATA_PATH):
        print(f"Warning: The path {DATA_PATH} does not exist.")
    else:
        dataset = TiDataset(DATA_PATH) 
        print(f"Dataset ready. Training on {len(dataset)} annotated images.")
        
      

In [ ]:
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt  

# Create Model
model = smp.Unet(
    encoder_name="resnet18",        
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=1,                      
    activation=None  
)

# Training Settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)

epoch_history = []
loss_history = []


dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Training Loop
epochs = 100
print(f"Starting Training on {device}...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for images, masks in dataloader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    epoch_history.append(epoch + 1)
    loss_history.append(avg_loss)
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

print("✅ Training Complete.")
torch.save(model.state_dict(), "titanium_unet_weights.pth")

plt.figure(figsize=(10, 5))
plt.plot(epoch_history, loss_history, label='Training Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Dice Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Pick a random patch to test

test_img_path = f"{DATA_PATH}/patches/patch_3.jpg" # Change to any patch
image = cv2.imread(test_img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Preprocess
x = image_rgb.transpose(2, 0, 1).astype('float32') / 255.0
x = torch.from_numpy(x).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    pred_mask = model(x)
    pred_mask = (pred_mask > 0.5).float().cpu().numpy()[0][0]

# Display
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original SEM")
plt.subplot(1, 2, 2)
plt.imshow(pred_mask, cmap='gray')
plt.title("U-net Prediction (Alpha Phase)")
plt.show()